In [1]:
import requests
import pandas as pd

# 1-day before weather forecasts
url = "https://previous-runs-api.open-meteo.com/v1/forecast"
params = {
    # 2107 Arbuckle, CA location
    "latitude": 38.9963,
    "longitude": -122.1341,
    "start_date": "2022-03-23",
    "end_date": "2024-10-31",
    # Use 'minutely_15' instead of 'hourly'
    "minutely_15": [
        "temperature_2m",
        "relative_humidity_2m",
        "dew_point_2m",
        "apparent_temperature",
        "wind_speed_10m",
        "wind_speed_80m",
        "wind_direction_10m",
        "wind_direction_80m",
        "wind_gusts_10m",
        "shortwave_radiation",
        "direct_radiation",
        "direct_normal_irradiance",
        "diffuse_radiation",
        "global_tilted_irradiance",
        "sunshine_duration",
        "precipitation",
        "snowfall",
        "rain",
        "cape",
        "visibility",
        "weather_code",
        "cloud_cover",
        "pressure_msl",
        "surface_pressure"
    ], 
    "models": "gfs_seamless",
    "previous_day": 1 
}

data = requests.get(url, params=params).json()
weather_df = pd.DataFrame(data['minutely_15']).rename(columns={"time": "measured_on"})

###########################
# Modify the file path used by pd.read_csv()


# meter_train date range: 2017-01-01 - 2023/11/09
meter_train = pd.read_csv("2107_meter_15m_data.csv")


# meter_2024 date range: 2024/01/01 - 2024/10/31
meter_2024 = pd.read_csv("2107_meter_15m_data_2024.csv")

# ensure same datetime type
weather_df["measured_on"] = pd.to_datetime(weather_df["measured_on"])
meter_train["measured_on"] = pd.to_datetime(meter_train["measured_on"])
meter_2024["measured_on"] = pd.to_datetime(meter_2024["measured_on"])


# df_train date range: 2022/03/23 - 2023/11/09
df_train = meter_train.merge(weather_df, on="measured_on", how="left").dropna()
# df_2024 date range: 2024/01/01 - 2024/10/31
df_2024 = meter_2024.merge(weather_df, on="measured_on", how="outer").dropna()

val_end = pd.Timestamp("2024-05-31 23:59:59")
# df_val date range: 2024-01-01 - 2024-05-31
df_val = df_2024[df_2024["measured_on"] <= val_end].copy()
# df_test date range: 2024-06-01 - 2024-10-31
df_test = df_2024[df_2024["measured_on"] > val_end].copy()

# Use df_train, df_val, df_test for train, validation(walk forward validation) and test.

In [2]:
print(df_train.head())

               measured_on  meter_revenue_grade_ac_output_meter_149578  \
182855 2022-03-23 00:00:00                                         0.0   
182856 2022-03-23 00:15:00                                         0.0   
182857 2022-03-23 00:30:00                                         0.0   
182858 2022-03-23 00:45:00                                         0.0   
182859 2022-03-23 01:00:00                                         0.0   

        temperature_2m  relative_humidity_2m  dew_point_2m  \
182855            29.2                  25.0           7.2   
182856            29.1                  25.0           7.1   
182857            28.8                  26.0           7.4   
182858            28.5                  27.0           7.7   
182859            28.1                  27.0           7.3   

        apparent_temperature  wind_speed_10m  wind_speed_80m  \
182855                  25.8            18.8            23.1   
182856                  25.8            17.5          